In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from impulse import PTSampler
from impulse.experimental import make_product_space_sampler, BirthDeathProductSpace
from impulse import model_visitation_stats, bayes_factor_from_chain

%load_ext autoreload
%autoreload 2

<![CDATA[## Product Space Sampling: 1 to 3 Sinusoids

This example demonstrates **trans-dimensional MCMC** using the product space method
via `BirthDeathProductSpace` and `make_product_space_sampler()`.

We simultaneously infer:
- The **number** of sinusoidal components (1, 2, or 3)
- The **parameters** (amplitude, frequency, phase) of each component

`BirthDeathProductSpace` handles:
- Birth/death proposals with correct Hastings ratios
- Model-index jump and source-swap proposals
- Parameter groups (one per source, model index excluded)
- Prior enforcement on all source parameters (active + inactive)
- Initial position drawing from the prior]]>

### Generate synthetic data

We inject **2 sinusoids** into Gaussian noise to test whether the sampler recovers the correct model.

In [ ]:
rng = np.random.default_rng(42)

N_pts = 200
t = np.linspace(0, 2 * np.pi, N_pts)
sigma = 1.0

# True signal: 2 sinusoids
A_true = [2.5, 1.2]
f_true = [0.8, 2.1]
phi_true = [0.5, 1.8]
n_true = len(A_true)

signal = np.zeros(N_pts)
for A, f, phi in zip(A_true, f_true, phi_true):
    signal += A * np.sin(2 * np.pi * f * t + phi)

data = signal + sigma * rng.standard_normal(N_pts)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(t, data, '.', alpha=0.5, label='Data')
plt.plot(t, signal, 'r-', lw=2, label='True signal (2 sinusoids)')
plt.xlabel('Time')
plt.ylabel('Amplitude')
plt.legend()
plt.title('Synthetic data: 2 sinusoids + Gaussian noise')
plt.tight_layout()

<![CDATA[### Product space setup

The parameter vector has shape `(MAX_SOURCES * NUM_PARAMS + 1,)`:

```
[A1, f1, phi1, A2, f2, phi2, A3, f3, phi3, nmodel]
```

where `nmodel` is the model index:
- `nmodel = 0`: 1 sinusoid (only source 1 is active)
- `nmodel = 1`: 2 sinusoids (sources 1-2 are active)
- `nmodel = 2`: 3 sinusoids (sources 1-3 are active)

`BirthDeathProductSpace` manages the model index and calls the user-provided
likelihood/prior with source parameters only.]]>

In [ ]:
MAX_SOURCES = 3
NUM_PARAMS = 3  # (amplitude, frequency, phase) per source

# Prior bounds
A_MIN, A_MAX = 0.0, 5.0
F_MIN, F_MAX = 0.0, 3.0
PHI_MIN, PHI_MAX = 0.0, np.pi


def source_prior_draw(rng):
    """Draw one source's parameters from the prior."""
    return np.array([
        rng.uniform(A_MIN, A_MAX),
        rng.uniform(F_MIN, F_MAX),
        rng.uniform(PHI_MIN, PHI_MAX),
    ])


def log_prior(params):
    """Uniform prior on all source parameters (active + inactive).

    BirthDeathProductSpace passes all num_sources * num_params parameters
    to this function, so bounds are enforced on inactive sources too.
    """
    n_sources = len(params) // NUM_PARAMS
    for i in range(n_sources):
        A = params[i * NUM_PARAMS]
        f = params[i * NUM_PARAMS + 1]
        phi = params[i * NUM_PARAMS + 2]
        if not (A_MIN <= A <= A_MAX):
            return -np.inf
        if not (F_MIN <= f <= F_MAX):
            return -np.inf
        if not (PHI_MIN <= phi <= PHI_MAX):
            return -np.inf
    return 0.0


def log_likelihood(params):
    """Gaussian likelihood using active sinusoidal components."""
    n_sources = len(params) // NUM_PARAMS
    model = np.zeros(N_pts)
    for i in range(n_sources):
        A = params[i * NUM_PARAMS]
        f = params[i * NUM_PARAMS + 1]
        phi = params[i * NUM_PARAMS + 2]
        model += A * np.sin(2 * np.pi * f * t + phi)
    return -0.5 * np.sum(((data - model) / sigma) ** 2)

<![CDATA[### Build the RJMCMC sampler

`make_product_space_sampler()` automatically registers birth, death,
model-index jump, and source-swap proposals alongside the standard
AM / SCAM / DE proposals. Parameter groups (one per source, excluding
the model index) are configured automatically.]]>

In [ ]:
space = BirthDeathProductSpace(
    loglikelihood=log_likelihood,
    logprior=log_prior,
    num_sources=MAX_SOURCES,
    num_params=NUM_PARAMS,
    source_prior_draw=source_prior_draw,
)

sampler = make_product_space_sampler(
    space,
    ntemps=15,
    inf_temp=True,
    seed=42,
    outdir='./chains_product_space',
)

x0 = space.draw_initial_position(rng, nmodel=1)

print(f"Parameter space: {space.ndim} dimensions")
print(f"  {MAX_SOURCES} sources x {NUM_PARAMS} params = {MAX_SOURCES * NUM_PARAMS} source params + 1 model index")
print(f"Groups: {space.get_default_groups()}")
print(f"Initial model: nmodel={int(x0[-1])} ({int(x0[-1]) + 1} sinusoid(s))")

In [ ]:
sampler.sample(x0, num_iterations=100_000)

### Results

In [ ]:
chains = sampler.load_chain()
cold_chain = chains['samples'][0]
cold_lnlike = chains['lnlike'][0]
burn = 30_000

probs = space.model_posterior_probs(cold_chain, burn=burn)

print("=== Model Selection ===")
for k in range(MAX_SOURCES):
    print(f"  P({k + 1} sinusoid{'s' if k > 0 else ' '}) = {probs[k]:.3f}")

preferred = np.argmax(probs)
print(f"\nTrue number of sinusoids: {n_true}")
print(f"Most probable model:      {preferred + 1} sinusoid(s)")

In [ ]:
nmodel_samples = np.rint(cold_chain[burn:, -1]).astype(int)

print("=== Model Selection ===")
for k in range(MAX_SOURCES):
    count = np.sum(nmodel_samples == k)
    frac = count / len(nmodel_samples)
    print(f"  P({k + 1} sinusoid{'s' if k > 0 else ' '}) = {frac:.3f}  ({count} samples)")

preferred = np.argmax(np.bincount(nmodel_samples))
print(f"\nTrue number of sinusoids: {n_true}")
print(f"Most probable model:      {preferred + 1} sinusoid(s)")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 5))

axes[0].plot(cold_chain[:, -1], alpha=0.5, lw=0.5)
axes[0].axhline(n_true - 1, color='r', ls='--', label=f'True nmodel = {n_true - 1}')
axes[0].set_ylabel('nmodel')
axes[0].set_title('Model index trace')
axes[0].legend()

axes[1].plot(cold_lnlike, alpha=0.5, lw=0.5)
axes[1].set_ylabel('Log-likelihood')
axes[1].set_xlabel('Iteration')
axes[1].set_title('Log-likelihood trace')
plt.tight_layout()

nmodel_samples = np.rint(cold_chain[burn:, -1]).astype(int)
mask = nmodel_samples == preferred
preferred_samples = cold_chain[burn:][mask]

true_vals = [A_true, f_true, phi_true]
param_labels = ['A', 'f', 'phi']

print(f"=== Parameter estimates ({preferred + 1}-sinusoid model) ===")
for i in range(preferred + 1):
    print(f"\n  Source {i + 1}:")
    for j, name in enumerate(param_labels):
        idx = i * NUM_PARAMS + j
        med = np.median(preferred_samples[:, idx])
        lo = np.percentile(preferred_samples[:, idx], 16)
        hi = np.percentile(preferred_samples[:, idx], 84)
        line = f"    {name:>3s} = {med:.3f}  [{lo:.3f}, {hi:.3f}]"
        if i < n_true:
            line += f"  (true: {true_vals[j][i]:.3f})"
        print(line)

In [ ]:
mask = nmodel_samples == preferred
preferred_samples = cold_chain[burn:][mask]

true_vals = [A_true, f_true, phi_true]
param_labels = ['A', 'f', 'phi']

print(f"=== Parameter estimates ({preferred + 1}-sinusoid model) ===")
for i in range(preferred + 1):
    print(f"\n  Source {i + 1}:")
    for j, name in enumerate(param_labels):
        idx = i * NUM_PARAMS + j
        med = np.median(preferred_samples[:, idx])
        lo = np.percentile(preferred_samples[:, idx], 16)
        hi = np.percentile(preferred_samples[:, idx], 84)
        line = f"    {name:>3s} = {med:.3f}  [{lo:.3f}, {hi:.3f}]"
        if i < n_true:
            line += f"  (true: {true_vals[j][i]:.3f})"
        print(line)

### Best-fit model vs data

In [ ]:
best_idx = np.argmax(cold_lnlike[burn:]) + burn
best_params = cold_chain[best_idx]
best_nmodel = int(np.rint(best_params[-1]))

best_model = np.zeros(N_pts)
for i in range(best_nmodel + 1):
    A = best_params[i * NUM_PARAMS]
    f = best_params[i * NUM_PARAMS + 1]
    phi = best_params[i * NUM_PARAMS + 2]
    best_model += A * np.sin(2 * np.pi * f * t + phi)

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].plot(t, data, '.', alpha=0.3, label='Data')
axes[0].plot(t, signal, 'r-', lw=2, label='True signal')
axes[0].plot(t, best_model, 'k--', lw=1.5, label=f'Best fit ({best_nmodel + 1} sinusoids)')
axes[0].legend()
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Data and model fits')

axes[1].plot(t, data - best_model, '.', alpha=0.3)
axes[1].axhline(0, color='k', ls='-', alpha=0.3)
axes[1].axhline(sigma, color='r', ls='--', alpha=0.5, label=f'+/- {sigma} sigma')
axes[1].axhline(-sigma, color='r', ls='--', alpha=0.5)
axes[1].set_xlabel('Time')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residuals')
axes[1].legend()
plt.tight_layout()

### Verify prior enforcement

Check that no samples (including inactive source parameters) ever violate
the prior bounds.

In [ ]:
print("=== Prior enforcement check ===")
post_burn = cold_chain[burn:]
all_ok = True
for i in range(MAX_SOURCES):
    A_vals = post_burn[:, i * NUM_PARAMS]
    f_vals = post_burn[:, i * NUM_PARAMS + 1]
    phi_vals = post_burn[:, i * NUM_PARAMS + 2]
    violations = (
        np.any(A_vals < A_MIN) or np.any(A_vals > A_MAX)
        or np.any(f_vals < F_MIN) or np.any(f_vals > F_MAX)
        or np.any(phi_vals < PHI_MIN) or np.any(phi_vals > PHI_MAX)
    )
    status = 'VIOLATION' if violations else 'OK'
    if violations:
        all_ok = False
    print(
        f"  Source {i + 1}:  "
        f"A in [{A_vals.min():.3f}, {A_vals.max():.3f}],  "
        f"f in [{f_vals.min():.3f}, {f_vals.max():.3f}],  "
        f"phi in [{phi_vals.min():.3f}, {phi_vals.max():.3f}]  "
        f"- {status}"
    )

nmodel_vals = np.rint(post_burn[:, -1]).astype(int)
valid = np.all((nmodel_vals >= 0) & (nmodel_vals < MAX_SOURCES))
if not valid:
    all_ok = False
print(f"  nmodel:   all in {{0..{MAX_SOURCES - 1}}} = {valid}")
print(f"\nAll priors respected: {all_ok}")